In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("Gaming Analytics - Star Schema Validation")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.local.dir", "D:/spark_temp")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)


fact = spark.read.parquet(
    "../data/gold/fact_game_analysis"
)

dim_game = spark.read.parquet(
    "../data/gold/dim_game"
)

dim_genre = spark.read.parquet(
    "../data/gold/dim_genre"
)


print("Missing game keys:")

fact.join(
    dim_game,
    "game_key",
    "left_anti"
).count()


print("Missing genre keys:")

fact.join(
    dim_genre,
    "genre_key",
    "left_anti"
).count()

Missing game keys:
Missing genre keys:


0

In [7]:
print("\n========== FIND MISSING GENRE KEY ==========\n")

fact.filter(
    col("genre_key").isNull()
).show(
    truncate=False
)


========== FIND MISSING GENRE KEY ==========

+--------+------+---------+-------------+-------------+---------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+--------------------+-----------+---------------+
|game_key|app_id|genre_key|publisher_key|developer_key|price_key|price|average_playtime|positive_ratings|negative_ratings|total_reviews|positive_reviews|negative_reviews|recommendation_rate|average_review_score|rawg_rating|rawg_metacritic|
+--------+------+---------+-------------+-------------+---------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+--------------------+-----------+---------------+
+--------+------+---------+-------------+-------------+---------+-----+----------------+----------------+----------------+-------------+----------------+----------------+-------------------+--------------------+-----------+----------

In [8]:
fact.groupBy(
    "game_key",
    "genre_key"
).count().filter(
    col("count") > 1
).count()

0

In [9]:
fact.select(
    "recommendation_rate",
    "average_review_score"
).describe().show()

+-------+-------------------+--------------------+
|summary|recommendation_rate|average_review_score|
+-------+-------------------+--------------------+
|  count|              76462|               76462|
|   mean|0.21516816196280436| 0.13133519278612824|
| stddev|0.35602079830774336|  0.3365466714464586|
|    min|                0.0|                -1.0|
|    max|                1.0|                 1.0|
+-------+-------------------+--------------------+

